# Bonds and fixed income

**Prerequisites:** **05** (pricing fundamentals). Price **vanilla fixed**, a **floating (FRN-style) bond**, and an **amortizing** bond; request rate risk and spread-style metrics where the registry exposes them.


## Concept

- **Vanilla fixed:** PV from discount curve + schedule.
- **FRN:** floating leg needs a **forward curve** for projected index fixings.
- **Amortizing:** principal path feeds coupon bases.
- **Risk:** `dv01`, **modified duration**, **convexity**; **Z-spread** / **I-spread** appear when the pricer stack supports them for your spec.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, ForwardCurve, MarketContext

print("Imports OK (bonds).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

fixed_bond = {
    "type": "bond",
    "spec": {
        "id": "UST-FIXED-10Y",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2034-01-15",
        "discount_curve_id": "USD-OIS",
        "accrual_method": "Linear",
        "settlement_days": 1,
        "ex_coupon_days": 0,
        "cashflow_spec": {
            "Fixed": {
                "coupon_type": "Cash",
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "end_of_month": False,
                "payment_lag_days": 0,
                "rate": "0.0425",
                "stub": "None",
            }
        },
        "call_put": None,
        "credit_curve_id": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {"quoted_clean_price": 98.75},
    },
}

frn_bond = {
    "type": "bond",
    "spec": {
        "id": "FRN-USD-SOFR",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "accrual_method": "Linear",
        "settlement_days": 2,
        "ex_coupon_days": 0,
        "cashflow_spec": {
            "Floating": {
                "coupon_type": "Cash",
                "freq": {"count": 3, "unit": "months"},
                "stub": "ShortFront",
                "rate_spec": {
                    "index_id": "USD-SOFR-3M",
                    "spread_bp": "150",
                    "gearing": "1",
                    "gearing_includes_spread": True,
                    "floor_bp": "0",
                    "all_in_floor_bp": None,
                    "cap_bp": None,
                    "index_cap_bp": None,
                    "dc": "Act360",
                    "bdc": "modified_following",
                    "calendar_id": "weekends_only",
                    "fixing_calendar_id": None,
                    "reset_freq": {"count": 3, "unit": "months"},
                    "reset_lag_days": 2,
                    "payment_lag_days": 0,
                    "end_of_month": False,
                },
            }
        },
        "call_put": None,
        "credit_curve_id": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {},
    },
}

amort_bond = {
    "type": "bond",
    "spec": {
        "id": "AMORT-NOTE",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "accrual_method": "Linear",
        "settlement_days": 2,
        "ex_coupon_days": 0,
        "cashflow_spec": {
            "Amortizing": {
                "base": {
                    "Fixed": {
                        "coupon_type": "Cash",
                        "freq": {"count": 6, "unit": "months"},
                        "dc": "Thirty360",
                        "bdc": "following",
                        "calendar_id": "weekends_only",
                        "end_of_month": False,
                        "payment_lag_days": 0,
                        "rate": "0.045",
                        "stub": "ShortFront",
                    }
                },
                "schedule": {"LinearTo": {"final_notional": {"amount": "250000", "currency": "USD"}}},
            }
        },
        "call_put": None,
        "credit_curve_id": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {},
    },
}

for label, payload in (("fixed", fixed_bond), ("frn", frn_bond), ("amort", amort_bond)):
    try:
        validate_instrument_json(json.dumps(payload))
        print(label, "JSON OK")
    except Exception as exc:
        print(label, "validate:", type(exc).__name__, exc)

bond_fixed_json = json.dumps(fixed_bond)
bond_frn_json = json.dumps(frn_bond)
bond_amort_json = json.dumps(amort_bond)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    [(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043), (10.0, 0.041)],
    base_date=AS_OF,
    day_count="act_360",
)
mc = MarketContext().insert(ois).insert(sofr)
market_json = mc.to_json()
print("curves in snapshot:", len(json.loads(market_json)["curves"]))


## Pricing


In [ ]:
for label, bjson in (
    ("fixed", bond_fixed_json),
    ("frn", bond_frn_json),
    ("amort", bond_amort_json),
):
    try:
        out = price_instrument_with_metrics(bjson, market_json, AS_OF_STR, model="discounting")
        print(label, ValuationResult.from_json(out))
    except Exception as exc:
        print(label, "price:", type(exc).__name__, exc)


## Metrics


In [ ]:
metrics_req = ["dv01", "duration_mod", "convexity", "z_spread", "i_spread"]
for label, bjson in (("fixed", bond_fixed_json), ("frn", bond_frn_json)):
    try:
        out = price_instrument_with_metrics(
            bjson, market_json, AS_OF_STR, model="discounting", metrics=metrics_req
        )
        vr = ValuationResult.from_json(out)
        print("—", label)
        for m in metrics_req:
            v = vr.get_metric(m)
            if v is not None:
                print(f"  {m}: {v}")
    except Exception as exc:
        print(label, "metrics:", type(exc).__name__, exc)


## Takeaways

- **One pipeline** prices fixed, float, and amortizing bonds once curves are wired (`USD-OIS` + `USD-SOFR-3M` for FRNs).
- **Metrics** are registry-driven: missing names are omitted rather than erroring.
- **Quoted clean** on the fixed bond pins a level so the engine can imply yields/spreads consistently with desk practice.
